# Notebook 2: Attention Allocation

Split **attention hours** across `create`, `distribute`, `learn`, `recover`. Effective clarity and reach depend on allocation + recovery debt.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("dark_background")
plt.rcParams["figure.figsize"] = (10, 5)


@dataclass
class Allocation:
    create: float
    distribute: float
    learn: float
    recover: float


def clamp_alloc(a: Allocation, total: float) -> Allocation:
    s = a.create + a.distribute + a.learn + a.recover
    if s <= 0:
        return Allocation(total * 0.25, total * 0.25, total * 0.25, total * 0.25)
    k = total / s
    return Allocation(a.create * k, a.distribute * k, a.learn * k, a.recover * k)


def productivity(alloc: Allocation) -> float:
    # recover boosts effective create quality; starving recover hurts
    base = 0.4 * alloc.create + 0.35 * alloc.distribute + 0.15 * alloc.learn
    recovery_bonus = min(alloc.recover, 3.0) * 0.08
    recovery_penalty = max(0.0, 2.0 - alloc.recover) * 0.15
    return max(0.1, base + recovery_bonus - recovery_penalty)


total_hours = 6.0
base = Allocation(2.0, 1.5, 1.0, 1.5)
alloc = clamp_alloc(base, total_hours)
eff = productivity(alloc)
alloc, eff


In [ ]:
# sweep: shift 0.5h from create -> distribute
xs = np.linspace(-1.5, 1.5, 31)
effs = []
for dx in xs:
    a = Allocation(
        base.create - dx,
        base.distribute + dx,
        base.learn,
        base.recover,
    )
    a2 = clamp_alloc(a, total_hours)
    effs.append(productivity(a2))

fig, ax = plt.subplots()
ax.plot(xs, effs)
ax.set_xlabel("shift hours from create → distribute (+)")
ax.set_ylabel("effective productivity index")
plt.show()
